In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://DESKTOP-AEBMQ4R.localdomain:7077") \
    .appName('test') \
    .getOrCreate()

26/03/09 02:58:01 WARN Utils: Your hostname, DESKTOP-AEBMQ4R resolves to a loopback address: 127.0.1.1; using 172.22.200.250 instead (on interface eth0)
26/03/09 02:58:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 02:58:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 02:58:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

26/03/09 02:59:57 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/03/09 03:00:12 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/03/09 03:00:27 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/03/09 03:00:42 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/03/09 03:00:57 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/03/09 03:01:12 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th

In [3]:
# Si esto te da un número mayor a cero, ignora el Warning
print(df_green.count())

[Stage 1:>                                                          (0 + 8) / 8]

2304517


In [4]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [6]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [7]:
print(df_yellow.count())

[Stage 6:========================>                                 (5 + 7) / 12]

39649199


In [8]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [10]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [11]:
from pyspark.sql import functions as F

In [12]:
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

In [13]:
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [14]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [15]:
df_trips_data.groupBy('service_type').count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [16]:
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

In [18]:
df_trips_data.registerTempTable('trips_data')

/home/axl/_infotech/_formacion/datatalkclub/de-zoomcamp-axl/06-batch-axl/.venv/lib/python3.13/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [19]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 12:===============================================>        (17 + 3) / 20]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [20]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [23]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [25]:
df_final = spark.read.parquet('data/report/revenue/')

# Corregido: añadimos "_monthly_" al nombre de la columna
df_final.sort("revenue_monthly_total_amount", ascending=False).show(5)

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         132

In [21]:
# Convertimos el resultado a Pandas para una visualización profesional
df_result.limit(10).toPandas()

,revenue_zone,revenue_month,service_type,revenue_monthly_fare,revenue_monthly_extra,revenue_monthly_mta_tax,revenue_monthly_tip_amount,revenue_monthly_tolls_amount,revenue_monthly_improvement_surcharge,revenue_monthly_total_amount,revenue_monthly_congestion_surcharge,avg_monthly_passenger_count,avg_monthly_trip_distance
0,250,2020-02-01,green,15359.96,1282.50,117.5,56.01,590.32,180.0,17598.44,11.00,1.239496,4.962811
1,158,2020-02-01,green,124.36,8.25,0.5,0.00,2.80,0.9,136.81,NaN,NaN,11.090000
2,152,2020-02-01,green,37630.82,1541.25,1398.5,2836.66,538.37,903.9,45984.60,1336.25,1.224006,2.726403
3,243,2019-12-01,green,13.99,0.00,0.0,1.00,0.00,0.3,15.29,0.00,1.000000,0.000000
4,20,2020-01-01,green,11375.42,681.00,131.0,90.62,479.86,147.3,12923.20,11.00,1.229787,4.872312
5,12,2020-01-01,green,161.01,5.50,1.0,0.00,12.24,1.2,180.95,NaN,NaN,0.715000
6,248,2019-12-01,green,10.00,0.50,0.5,0.00,0.00,0.3,11.30,0.00,1.000000,2.900000
7,182,2020-02-01,green,19047.14,1493.00,183.5,209.76,676.00,267.0,21901.75,16.50,1.275053,4.035102
8,37,2020-02-01,green,23375.71,1982.75,229.0,428.95,527.51,284.1,26942.42,104.25,1.231806,4.494921
9,62,2020-02-01,green,20893.55,1908.50,192.0,212.17,124.69,262.5,23661.76,13.75,1.124555,3.892513
